# 🏆 Leaderboard · Task 3 · M4 Hourly forecasting

Starter for Task 3 of the [Leaderboard](Leaderboard.md): forecast the next 48 hours of 414 hourly series, scored by **MASE** (lower is better).

| | |
|---|---|
| **Data** | the **Hourly** subset of the M4 forecasting competition: 414 series, 700–960 observations each ([M4-methods repository](https://github.com/Mcompetitions/M4-methods)) |
| **Split** | the official M4 holdout: forecast the **48 hours** after each training series. Validation = the last 48 hours of each training series |
| **Metric** | MASE with seasonal period m = 24, averaged over the 414 series (the M4 definition) |
| **Baseline** | seasonal naive (repeat the last 24 hours): **1.1932** |
| **Rules** | one forecast per series from its own training history (other series may be used: global models are fine) · no test values · no external data |

MASE divides your forecast MAE by the in-sample MAE of the seasonal naive one-day-ahead forecast. Below 1 means "better than repeating yesterday, on the series' own scale", so series of very different magnitudes can be averaged.


**How to use this notebook**
1. Run everything once as delivered: it downloads the data, checks the split is the official one, and reproduces the baseline.
2. Write your model in **`my_model`** (section 4) and set `METHOD` to a short description.
3. Iterate on the **validation** score (section 5) as often as you like.
4. When you are happy, run section 6 **once**: it trains on the full training data, scores the test set, and prints a row for [Leaderboard](Leaderboard.md).
5. Save a copy of the notebook for your attempt if you want the row's link to point to the exact code (*File → Save as*, e.g. `Leaderboard - MNIST - small CNN.ipynb`).

Cells print ✅ / ❌ / ⏳ instead of raising, so the notebook always runs top to bottom. Saved without outputs.

In [ ]:
import datetime, hashlib, pathlib, re, time, urllib.request
import numpy as np

DATA = pathlib.Path("leaderboard-data")   # downloads are cached here (git-ignored)
DATA.mkdir(exist_ok=True)

def check(ok, msg_ok, msg_bad):
    print(("✅ " + msg_ok) if ok else ("❌ " + msg_bad))

def download(url, name):
    f = DATA / name
    if not f.exists():
        print("downloading", url)
        urllib.request.urlretrieve(url, f)
    return f

def leaderboard_best(task, higher_is_better, path="Leaderboard.md"):
    """Best score logged so far in the Leaderboard.md section whose heading contains `task`."""
    p = pathlib.Path(path)
    if not p.exists():
        return None
    best = None
    for sec in re.split(r"^## ", p.read_text(encoding="utf-8"), flags=re.M)[1:]:
        if task not in sec.splitlines()[0]:
            continue
        for line in sec.splitlines():
            cells = [c.strip() for c in line.strip().strip("|").split("|")]
            if len(cells) < 3 or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", cells[0]):
                continue
            try:
                score = float(cells[1])
            except ValueError:
                continue
            if best is None or (score > best[0] if higher_is_better else score < best[0]):
                best = (score, cells[2])
    return best

def report(task, score, method, baseline, higher_is_better, notebook, digits=4):
    """Compare a test score with the baseline and your best so far, and print a row for Leaderboard.md."""
    better = (lambda a, b: a > b) if higher_is_better else (lambda a, b: a < b)
    print(f"Test score: {score:.{digits}f}")
    check(better(score, baseline), f"beats the baseline ({baseline})", f"does not beat the baseline ({baseline}) yet")
    best = leaderboard_best(task, higher_is_better)
    if best:
        check(better(score, best[0]), f"new personal best! Previous best: {best[0]} ({best[1]})",
              f"not a new best. Current best: {best[0]} ({best[1]})")
    print("\nPaste this row at the bottom of the table in Leaderboard.md:")
    print(f"| {datetime.date.today()} | {score:.{digits}f} | {method} | [notebook]({notebook.replace(' ', '%20')}) |")

import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/"
tr = pd.read_csv(download(BASE_URL + "Train/Hourly-train.csv", "Hourly-train.csv"), index_col=0)
te = pd.read_csv(download(BASE_URL + "Test/Hourly-test.csv", "Hourly-test.csv"), index_col=0)
IDS = list(tr.index)
train = {k: r.dropna().to_numpy(float) for k, r in tr.iterrows()}   # series have different lengths
test = {k: r.to_numpy(float) for k, r in te.iterrows()}
H, M = 48, 24   # horizon and seasonal period
print(len(IDS), "series; lengths", min(map(len, train.values())), "to", max(map(len, train.values())))

## 1. The fixed split
The validation split mimics the real one: hide the last 48 hours of every training series and forecast them from the rest.

In [ ]:
dev_train = {k: v[:-H] for k, v in train.items()}   # fit here while developing
dev_val = {k: v[-H:] for k, v in train.items()}     # ...and compare ideas here

fp = hashlib.sha256(np.round(te.to_numpy(float), 3).tobytes()).hexdigest()[:12]
check(fp == "6aae7e9e22c0" and len(IDS) == 414 and list(te.index) == IDS,
      "data and split are the official ones", f"unexpected data (fingerprint {fp}): scores are not comparable")

## 2. The official metric
A forecast is a dict `series id → array of 48 values`. The scale of each series comes from its own history only.

In [ ]:
def mase(history, actual, forecast, m=M):
    scale = np.mean(np.abs(history[m:] - history[:-m]))   # in-sample MAE of seasonal naive
    return np.mean(np.abs(actual - forecast)) / scale

def mean_mase(histories, actuals, forecasts):
    missing = [k for k in IDS if k not in forecasts]
    assert not missing, f"no forecast for {len(missing)} series, e.g. {missing[:3]}"
    for k in IDS:
        assert np.shape(forecasts[k]) == (H,), f"{k}: expected {H} values, got {np.shape(forecasts[k])}"
        assert np.all(np.isfinite(forecasts[k])), f"{k}: forecast contains NaN/inf"
    return float(np.mean([mase(histories[k], actuals[k], np.asarray(forecasts[k], float)) for k in IDS]))

def val_mase(forecasts):
    """Development score: forecasts made from dev_train, scored on the last 48 training hours."""
    return mean_mase(dev_train, dev_val, forecasts)

def m4_mase(forecasts):
    """Official Task 3 metric: forecasts made from the full training series, scored on the M4 test set."""
    return mean_mase(train, test, forecasts)

# Self-test: a perfect forecast scores 0; missing every hour by exactly the series' scale scores 1
shifted = {k: test[k] + np.mean(np.abs(train[k][M:] - train[k][:-M])) for k in IDS}
check(m4_mase(test) == 0.0 and abs(m4_mase(shifted) - 1) < 1e-9, "metric self-test passed", "metric self-test failed")

## 3. Baseline: seasonal naive

In [ ]:
BASELINE = 1.1932

def seasonal_naive(histories):
    return {k: np.tile(v[-M:], H // M) for k, v in histories.items()}

print(f"validation MASE {val_mase(seasonal_naive(dev_train)):.4f}")
base_mase = m4_mase(seasonal_naive(train))
print(f"test MASE       {base_mase:.4f}")
check(round(base_mase, 4) == BASELINE, f"baseline reproduced ({BASELINE})", f"baseline differs from {BASELINE}")

## 4. Your model ✏️
`my_model(histories)` gets a dict `series id → 1-D array` and returns a dict `series id → 48 forecasts`. Return `None` until you have something.

In [ ]:
METHOD = "my model"   # e.g. "global ridge on 336 scaled lags"

def my_model(histories):
    # TODO: return {series_id: np.array of 48 forecasts} for every series in histories
    return None

<details>
<summary><b>Example submission</b> (shows the interface; try your own idea first)</summary>

A **global** linear model: one ridge regression trained on windows from all 414 series. Each window of the last 168 hours is scaled by the series' seasonal-naive MAE, and the target is the **correction to seasonal naive** rather than the raw values. Scores **0.9619** in a few seconds.

```python
from sklearn.linear_model import Ridge

METHOD = "global ridge, 168 lags, predicts correction to seasonal naive"

def my_model(histories, lags=168, step=12):
    def features(x, scale):
        return (x - x[-M:].mean()) / scale
    X, Y = [], []
    for v in histories.values():
        scale = np.mean(np.abs(v[M:] - v[:-M]))
        for end in range(len(v) - H, lags - 1, -step):      # training windows inside the history
            x, y = v[end - lags:end], v[end:end + H]
            X.append(features(x, scale)); Y.append((y - np.tile(x[-M:], H // M)) / scale)
    model = Ridge(alpha=100).fit(np.array(X), np.array(Y))
    forecasts = {}
    for k, v in histories.items():
        scale = np.mean(np.abs(v[M:] - v[:-M])); x = v[-lags:]
        forecasts[k] = np.tile(x[-M:], H // M) + model.predict(features(x, scale)[None])[0] * scale
    return forecasts
```

</details>

**Ideas to try:** tune the number of lags, `alpha` and the window step on validation, add calendar features (hour of day), direct multi-output vs recursive forecasting, gradient boosting on lag features, ensembling with seasonal naive, a small MLP or N-BEATS-style network.
Two things that do **not** work, measured on this setup: averaging the same hour over the last two days (1.3772) and a ridge regression that predicts raw values from lags (about 1.73), both worse than the baseline.
See [[Time Series Forecasting]], [[Classical Forecasting Models]], [[Time Series Validation and Features]], [[Deep Learning for Time Series]].

## 5. Validation score (use as often as you like)

In [ ]:
t = time.time()
fc_val = my_model(dev_train)
if fc_val is None:
    print("⏳ my_model returns None: write your model in section 4")
else:
    print(f"validation MASE {val_mase(fc_val):.4f} ({time.time() - t:.0f} s)")

## 6. Test score (once per idea)
Refits on the full training series, scores the M4 test set and compares with the baseline and with your best row in `Leaderboard.md`.

In [ ]:
fc_test = my_model(train)
if fc_test is None:
    print("⏳ my_model returns None: nothing to submit yet")
else:
    report("M4 Hourly", m4_mase(fc_test), METHOD, BASELINE, higher_is_better=False,
           notebook="Leaderboard - M4 Hourly.ipynb")